In [16]:
from google.colab import drive
drive.mount("/content/drive")

import glob
import pandas as pd

base = "/content/drive/MyDrive/ifood_marketing_analytics"
files = glob.glob(base + "/**/*.csv", recursive=True)
files

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


['/content/drive/MyDrive/ifood_marketing_analytics/data/ifood_df.csv',
 '/content/drive/MyDrive/ifood_marketing_analytics/outputs/ifood_with_segments.csv',
 '/content/drive/MyDrive/ifood_marketing_analytics/outputs/segment_profile.csv',
 '/content/drive/MyDrive/ifood_marketing_analytics/outputs/target_list_top200.csv',
 '/content/drive/MyDrive/ifood_marketing_analytics/outputs/ifood_scored_all_customers.csv']

In [17]:
path = base + "/outputs/ifood_with_segments.csv"
df = pd.read_csv(path)

print("Loaded:", path)
print("Shape:", df.shape)
df.head()

Loaded: /content/drive/MyDrive/ifood_marketing_analytics/outputs/ifood_with_segments.csv
Shape: (2205, 45)


,Income,Kidhome,Teenhome,Recency,MntWines,MntFruits,MntMeatProducts,MntFishProducts,MntSweetProducts,MntGoldProds,...,education_PhD,MntTotal,MntRegularProds,AcceptedCmpOverall,CustomerID,TotalKids,TotalPurchases,DealShare,Segment,SegmentName
0,58138.0,0,0,58,635,88,546,172,88,88,...,0,1529,1441,0,1,0,25,0.120000,2,Quiet Value Regulars
1,46344.0,1,1,38,11,1,6,2,1,6,...,0,21,15,0,2,2,6,0.333333,1,Deal Driven Browsers
2,71613.0,0,0,26,426,49,127,111,21,42,...,0,734,692,0,3,0,21,0.047619,2,Quiet Value Regulars
3,26646.0,1,0,26,11,4,20,10,3,5,...,0,48,43,0,4,1,8,0.250000,1,Deal Driven Browsers
4,58293.0,1,0,94,173,43,118,46,27,15,...,1,407,392,0,5,1,19,0.263158,2,Quiet Value Regulars


In [18]:
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import pandas as pd

df = df.copy()
df["CustomerID"] = np.arange(1, len(df) + 1)

df["TotalKids"] = df["Kidhome"] + df["Teenhome"]
df["TotalPurchases"] = df["NumDealsPurchases"] + df["NumWebPurchases"] + df["NumCatalogPurchases"] + df["NumStorePurchases"]
df["DealShare"] = np.where(df["TotalPurchases"] > 0, df["NumDealsPurchases"] / df["TotalPurchases"], 0)

seg_features = ["Income","Recency","MntTotal","TotalPurchases","DealShare","NumWebVisitsMonth","TotalKids","AcceptedCmpOverall"]
X = df[seg_features].copy().fillna(df[seg_features].median(numeric_only=True))

X_scaled = StandardScaler().fit_transform(X)

rows = []
for k in range(3, 9):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    rows.append([k, silhouette_score(X_scaled, labels)])

sil_table = pd.DataFrame(rows, columns=["k","silhouette"]).sort_values("silhouette", ascending=False)
sil_table

,k,silhouette
0,3,0.289600
1,4,0.247444
2,5,0.214166
3,6,0.203337
5,8,0.201125
4,7,0.199036


In [22]:
import os
import matplotlib.pyplot as plt
import seaborn as sns

assets_path = base + "/assets"
os.makedirs(assets_path, exist_ok=True)

print(f"Assets directory created or already exists at: {assets_path}")

Assets directory created or already exists at: /content/drive/MyDrive/ifood_marketing_analytics/assets


In [23]:
fig, ax = plt.subplots(figsize=(8, 6))
sns.histplot(df['Income'], kde=True, ax=ax)
ax.set_title('Distribution of Income')
ax.set_xlabel('Income')
ax.set_ylabel('Frequency')

# Define the filename and save the plot
plot_filename = os.path.join(assets_path, 'income_distribution.png')
plt.savefig(plot_filename)
plt.close(fig) # Close the figure to free up memory

print(f"Plot saved to: {plot_filename}")

Plot saved to: /content/drive/MyDrive/ifood_marketing_analytics/assets/income_distribution.png


In [19]:
best_k = int(sil_table.iloc[0]["k"])
km = KMeans(n_clusters=best_k, random_state=42, n_init=10)
df["Segment"] = km.fit_predict(X_scaled)

profile = df.groupby("Segment")[seg_features].mean()
profile["Size"] = df.groupby("Segment").size()
profile["ResponseRate"] = df.groupby("Segment")["Response"].mean()
profile.sort_values("ResponseRate", ascending=False)

,Income,Recency,MntTotal,TotalPurchases,DealShare,NumWebVisitsMonth,TotalKids,AcceptedCmpOverall,Size,ResponseRate
Segment,,,,,,,,,,
0,79978.795082,50.286885,1502.668033,21.012295,0.056361,3.192623,0.192623,1.799180,244,0.491803
1,36344.320470,48.564597,135.800336,9.718960,0.255678,6.644295,1.290268,0.112416,1192,0.109060
2,66306.193758,49.292588,926.360208,20.957087,0.103152,3.990897,0.659298,0.113134,769,0.107932


In [20]:
segment_names = {
    0: "Premium Loyalists",
    1: "Deal Driven Browsers",
    2: "Quiet Value Regulars"
}

df["SegmentName"] = df["Segment"].map(segment_names)

df[["CustomerID","Segment","SegmentName"]].head()

,CustomerID,Segment,SegmentName
0,1,2,Quiet Value Regulars
1,2,1,Deal Driven Browsers
2,3,2,Quiet Value Regulars
3,4,1,Deal Driven Browsers
4,5,2,Quiet Value Regulars


In [21]:
import os

out_base = "/content/drive/MyDrive/ifood_marketing_analytics/outputs"
os.makedirs(out_base, exist_ok=True)

df.to_csv(out_base + "/ifood_with_segments.csv", index=False)
profile.to_csv(out_base + "/segment_profile.csv")

print("Saved to:", out_base)

Saved to: /content/drive/MyDrive/ifood_marketing_analytics/outputs
